# Prompt Engineering Patterns — From Scratch

---

## 🏗️ Build | 🚢 Ship | 📤 Share

### 🏗️ Build
- A thin helper layer on top of OpenAI's Responses API so every task stays short
- Seven prompt patterns, each demonstrated on a consulting-flavored task
- Weak→strong comparisons for persona, few-shot, and retrieved-context so you see *why* each pattern matters
- A reasoning-budget readout (answers, token counts, latency) so you can measure the cost/quality tradeoff

### 🚢 Ship
Take two or three of these patterns and apply them to your group project's real prompts. Keep the before/after side-by-side so you can explain the improvement to a client.

### 📤 Share
Which pattern moved your task the most, and why do you think that was?

---

Every AI deliverable you build will live or die on its prompts. Before you reach for fine-tuning, retrieval infrastructure, or a new model, try prompting better — it is the fastest and cheapest lever you have. This notebook is the smallest version of that, and the bigger tools (prompt-management platforms, automated evals, agent frameworks) are the same ideas scaled up.

**Estimated time:** 25–35 minutes

---

## Setup

Copy `.env.template` to `.env` at the repo root and fill in your credentials — `OPENAI_API_KEY` and `LLM_MODEL` for OpenAI, or one of the annotated provider blocks for Azure OpenAI, Bedrock, Anthropic, or anything else.

This notebook works on the **Responses API** (`respond()` and `structured()`), the modern surface for gpt-5-era models. Those two functions come from `helpers/llm_call.py`, which is the only file in the repo that knows where your models live: point it at your endpoint once and all seventeen modules follow. Run `python -m helpers.llm_call` to see what your environment resolves to.

On providers without a native Responses API, `structured()` falls back to JSON-schema prompting and still returns a validated object.

> **Reasoning effort** (`effort=` in the `ask()` helper) is a gpt-5+ / o-series feature and is ignored gracefully on older models.

In [1]:
import time, json
from pathlib import Path
from IPython.display import display, Markdown

import sys; sys.path.insert(0, "..")          # repo root, for the shared helpers
from helpers.llm_call import model_name, resolve, respond, structured

# Which model to call, at which endpoint, with which key — all decided in
# helpers/llm_call.py, the one file in this repo that knows. Set LLM_MODEL in your .env
# (or LLM_MODEL=azure/<your-deployment>) and nothing in this notebook changes.
# This notebook is about *prompting*, so it never names a provider.
MODEL = model_name("chat")

# ── Thin helpers ────────────────────────────────────────────────────────────

def ask(input, instructions=None, effort=None, **kw):
    """
    One call against the Responses API.
    `input`        — a plain string or a list of role-message dicts
    `instructions` — the system / developer instruction string
    `effort`       — reasoning effort: None | 'low' | 'medium' | 'high'
                     (gpt-5+ models only; None = no explicit reasoning step)
    Returns the Response object, so the cells below can read .output_text and .usage.
    """
    return respond(input, instructions=instructions, effort=effort, **kw)

def show(resp) -> None:
    """Pretty-print a Response object's text to the notebook."""
    display(Markdown(resp.output_text))

print(f"✅ Model: {MODEL}  API base: {resolve('chat').base_url or 'default'}")

✅ Model: gpt-5-mini  API base: default


## Task 1 — Personas (instructions steer voice)

The `instructions` parameter is your system prompt. It doesn't change what the model *knows* — it steers *how* the model speaks. The same question gets a completely different response depending on the persona you set.

This is the first thing to reach for on any client-facing AI feature: who should this assistant sound like?

We'll ask the same EU AI Act question three ways:
1. No persona (raw default)
2. A terse senior partner
3. A patient mentor

In [2]:
AI_ACT_Q = "What should a product team do if its AI system may fall into a high-risk EU AI Act category?"

# ── Weak: no persona ─────────────────────────────────────────────────────────
print("=" * 60)
print("WEAK — no persona")
print("=" * 60)
show(ask(AI_ACT_Q))

WEAK — no persona


Short answer
1) Treat this as a compliance priority: immediately assess whether the system is high‑risk under the EU AI Act.  
2) If it likely is, pause or limit deployments that would place the product on the EU market until you can meet the high‑risk obligations.  
3) Put in place the required governance, technical controls and documentation (risk management, data governance, technical documentation, human oversight, testing, post‑market monitoring), and run the appropriate conformity assessment (internal or with a notified body). Consult legal counsel.

Practical next steps (ordered)

1) Confirm classification
- Perform a legal/technical classification: map the product’s intended purpose(s), user groups and deployment contexts against the AI Act’s high‑risk categories (e.g., safety components of products, critical infrastructure, education/employment, biometric ID, law enforcement, migration/asylum/border control, access to essential services, etc.).  
- Get legal counsel involved to reduce risk of misclassification.

2) Limit exposure while assessing
- If you can, suspend or restrict EU deployments or high‑risk use cases until you’re confident about compliance. Use feature flags, geofencing, or controlled pilot environments.

3) Build your compliance team and plan
- Assemble cross‑functional owners: product, engineering, legal/regulatory, privacy, security, QA, risk, and senior management.  
- Create a milestone plan for conformity assessment, documentation, testing, and certification.

4) Implement mandatory organizational and technical controls
- Risk management system: identify, evaluate and mitigate risks across lifecycle; keep evidence of hazard analyses and mitigation decisions.  
- Data governance: datasets used for training/validation must meet quality and representativeness requirements; document provenance and preprocessing.  
- Technical documentation: produce the technical file that explains system design, intended use, architecture, algorithms, datasets, performance metrics and limitations.  
- Record-keeping (logs): implement continuous logging to enable traceability and post‑market analysis.  
- Human oversight: define and implement measures and instructions so humans can reliably oversee, intervene or override the system.  
- Robustness, accuracy and cybersecurity: test performance, robustness to attacks, and adversarial inputs; maintain secure development lifecycle.  
- Transparency and user information: provide required instructions for use and appropriate disclosures to users (what the system does, limitations, risk warnings).  
- Training and competence: ensure staff operating or supervising the system are trained.

5) Testing, validation and conformity assessment
- Run thorough testing and validation using representative datasets. Produce statistical performance reports, fairness and bias testing, and stress/adversarial testing.  
- Determine which conformity assessment route applies: some high‑risk systems require third‑party assessment by a notified body; others allow internal control with appropriate documentation. Prepare for a notified body audit if required.  
- Prepare the EU declaration of conformity and CE marking if applicable.

6) Registration and post‑market obligations
- Register the product in the EU AI database if required.  
- Set up post‑market monitoring and an incident management process: track performance in the field, collect user feedback, evaluate new risks, and report serious incidents/non‑compliances to authorities as required.

7) Contracts, suppliers and lifecycle controls
- Update supplier contracts and SLAs to ensure upstream compliance (data providers, third‑party models, cloud providers). Require evidence for components you depend on.  
- Plan for continuous monitoring and periodic re‑assessment across the product lifecycle, including after model updates.

8) Communicate with stakeholders
- Inform senior management and put compliance budget/personnel in place.  
- If you have customers in the EU, communicate restrictions, changes and timelines transparently.

Documentation checklist (minimum items to prepare)
- Legal classification memo and intended‑use statement  
- Risk assessment and mitigation log  
- Data governance policy and dataset documentation (sources, cleaning, labeling, sampling)  
- Technical documentation / technical file (architecture, algorithms, training/validation details)  
- Testing reports (accuracy, robustness, bias/fairness)  
- Human oversight and user instructions / labels  
- Cybersecurity assessment and secure‑SDLC evidence  
- Post‑market monitoring plan and incident reporting SOPs  
- Evidence of conformity assessment (internal or notified body), EU declaration of conformity, and registration evidence

Who to involve
- Internal: product manager, engineering, ML engineers, QA, privacy officer/DPO, security, legal/regulatory, risk/compliance, customer success, ops.  
- External: EU regulatory counsel, notified body (if needed), external auditors/testing labs, third‑party experts for safety or bias testing.

Final note
The AI Act imposes substantial, ongoing obligations for high‑risk systems. Treat classification and conformity as a product priority and get legal/regulatory expertise involved early — technical fixes alone are not sufficient. If you want, I can: a) help draft a gap‑analysis checklist tailored to your system, b) walk through a classification questionnaire, or c) outline the documentation structure you’ll need. Which would you like?

In [3]:
# ── Strong: two distinct personas ────────────────────────────────────────────
SENIOR_PARTNER = (
    "You are a terse senior partner at a management-consulting firm. "
    "Give direct, actionable recommendations in three sentences or fewer. "
    "No hedging, no pleasantries."
)

PATIENT_MENTOR = (
    "You are a patient senior consultant mentoring a junior colleague. "
    "Explain your reasoning step by step, anticipate follow-up questions, "
    "and close with a concrete next action."
)

print("=" * 60)
print("STRONG — Persona A: terse senior partner")
print("=" * 60)
show(ask(AI_ACT_Q, instructions=SENIOR_PARTNER))

print()
print("=" * 60)
print("STRONG — Persona B: patient mentor")
print("=" * 60)
show(ask(AI_ACT_Q, instructions=PATIENT_MENTOR))

STRONG — Persona A: terse senior partner


Pause deployment and treat the system as high‑risk until you complete a formal classification and a conformity gap analysis against the EU AI Act (see Annexes II–III and applicable harmonised standards).  
Appoint a named compliance lead, implement required technical and organisational controls (data governance, risk management, human oversight, robustness, cybersecurity, logging), produce the technical documentation/technical file, and engage a notified body or conformity assessor to complete the conformity assessment and CE marking before market placement.  
Establish post‑market monitoring, incident reporting and record‑retention processes, register in the EU database, and obtain external legal/regulatory support for filings and regulator interactions.


STRONG — Persona B: patient mentor


Good question. Treat this as a triage + compliance project: you must quickly decide whether the system actually is “high‑risk” under the EU AI Act, and — if it is — put in place the organization, controls and evidence the law requires. I’ll walk you through the reasoning and give a concrete next action.

Step‑by‑step reasoning and recommended actions

1) Triage: do a quick classification
- Ask the core question: what is the system’s intended purpose and in which sector will it be deployed? The Act’s “high‑risk” scope depends on intended purpose + sector (see Annex III categories such as employment/recruitment, education, safety components of products, law enforcement, biometric identification, border control, health/medical devices, critical infrastructure, essential services, migration/asylum, administration of justice, etc.).  
- Action: produce a 1‑page “intended use” statement describing who will use the system, where, and what decisions it supports/automates. That document determines classification.

2) If it may be high‑risk, stop public roll‑out in the EU (or restrict) until you’ve assessed compliance
- Risk: deploying before compliance can cause regulatory enforcement, market bans, and reputational/legal exposure.  
- Action: issue an immediate pause or limited pilot (if needed for testing) in the EU market while you assess.

3) Assemble a cross‑functional project team
- Core members: legal/regulatory, product manager (owner of intended use), engineering/ML, data privacy, security, QA/testing, risk/compliance, operations, and senior sponsor. If you’re a non‑EU provider, include a plan for an EU legal representative.  
- Action: schedule a kickoff meeting within 48 hours.

4) Perform a gap analysis vs. high‑risk obligations
- Key obligations for high‑risk systems you must evaluate and then implement are: risk management system; data governance and dataset documentation; technical documentation and record‑keeping (design, training, performance, tests); transparency and user instructions; human oversight measures; robustness, accuracy, and cybersecurity testing; post‑market monitoring and incident reporting; conformity assessment (internal or by notified body) and CE marking/Declaration of Conformity; registration in the EU database; appoint EU representative if required.  
- Action: have legal and engineering produce a mapped checklist showing “implemented / partial / not implemented” for each obligation.

5) Start a documented risk management process immediately
- Identify hazards from misuse, failure modes, bias and harms to people. Assess risk levels and define mitigations and acceptance criteria. Risk management is continuous and must be integrated into development lifecycles.  
- Action: create an initial risk register (top 10 risks) and assign owners.

6) Remediate gaps that are typically non‑trivial
- Typical workstreams: create and quality‑control training/validation datasets (representativeness, labeling quality, governance); add logging and technical documentation; design human oversight and stop/fallback modes; establish security controls and robustness testing (adversarial, stress, distribution shift); write user instructions and warnings; implement continuous monitoring and reporting.  
- Some work (e.g., third‑party conformity assessment) will take weeks/months — plan accordingly.

7) Determine conformity assessment route and timeline
- You’ll need to decide whether you can self‑assess (internal conformity assessment) or require a notified body. That determines how long certification will take and what evidence you must produce. Work with legal or a notified body early to avoid surprises.

8) Post‑market monitoring and incident handling
- Put in place mechanisms to log model outputs and incidents, monitor performance drift, analyse complaints, and report serious incidents to authorities per the Act’s requirements.

9) Consider design options to avoid classification (if appropriate)
- If being high‑risk is commercially onerous and legally avoidable, consider redesigning the product to change its intended purpose, scope, or how decisions are made (e.g., provide only non‑decision support information, avoid biometric ID, restrict use cases). But this must be a genuine change of intended use — not a cosmetic workaround.

10) Involve senior leadership and legal counsel
- This is an enterprise‑level compliance issue with potential market impact. Brief executives and legal counsel early and regularly.

Common follow‑up questions I anticipate (and short answers)
- How do I know for sure whether we are in Annex III? — Map your intended use to the Annex III categories. If unclear, ask external counsel or a notified body; get a written position.  
- How long does compliance and certification take? — It varies. Basic documentation and controls can take weeks; full third‑party conformity and certification often takes months. Start now.  
- What about existing deployed models? — You need post‑market monitoring and remediation; consider limiting exposures until compliant.  
- What if we’re headquartered outside the EU? — You’ll likely have to appoint an EU legal representative and comply with the same obligations for systems placed on the EU market.  
- Can we just stop selling in the EU? — Yes, that’s an option while you remediate, but consider contractual and customer impacts.

Deliverables you should build in the next 4–8 weeks
- 1‑page intended use statement and classification memo (high‑risk yes/no with rationale).  
- Gap analysis checklist against high‑risk obligations.  
- Initial risk register with mitigations and owners.  
- Timeline and resource plan for remediation and conformity assessment.  
- Draft technical documentation and dataset inventory (start a “technical file”).

Concrete next action (what I want you to do right now)
- Within 48 hours convene a 60‑minute kickoff with: product lead, lead ML engineer, legal/regulatory counsel, privacy/security lead, QA, and a senior sponsor. Agenda: review and sign off the 1‑page intended use statement, decide whether to pause EU rollout, and assign owners to create the classification memo and initial gap analysis to be delivered in 5 business days. I can help you draft the intended use statement and the gap checklist — say if you want a template.

If you want, I’ll draft the 1‑page intended use template and a tailored gap‑analysis checklist based on your product. Which product/system are we talking about, and what is its intended use in the EU?

Same model, same question, same facts — completely different communication styles. That's the persona pattern: **the instructions are the character brief**.

In production, personas live in versioned system-prompt templates so every team can deploy the same model with the brand voice locked in.

## Task 2 — Few-shot prompting (learning from examples)

The model already knows what words mean and how language works. Few-shot prompting teaches it a *task-specific pattern* it hasn't seen before — purely from examples in the prompt, no training required.

We'll see this in two steps:
1. A playful proof-of-concept with made-up words (`stimple`, `falbean`) — borrowed from the [GPT-3 few-shot paper](https://arxiv.org/abs/2005.14165) — to show the mechanism cleanly
2. A consulting-real task: tagging client requirements as *in-scope* or *out-of-scope*, where the labeling convention isn't something the model can guess from language alone

In the Responses API, few-shot examples are supplied as a **role-message list** — just like a real conversation: user turn → assistant turn → new user turn.

In [4]:
# ── Part A: made-up words (zero-shot vs. few-shot) ───────────────────────────

print("=" * 60)
print("WEAK — zero-shot (no examples for made-up words)")
print("=" * 60)
show(ask("Use the words 'stimple' and 'falbean' in a sentence."))

WEAK — zero-shot (no examples for made-up words)


The stimple moths fluttered around the falbean lantern, painting the garden with tiny, iridescent sparks.

In [5]:
# ── Few-shot: teach meanings via prior turns ──────────────────────────────────
few_shot_examples = [
    {"role": "user",      "content": "Something 'stimple' is good, high-quality, and well-functioning. "
                                     "Use 'stimple' in a sentence."},
    {"role": "assistant", "content": "'Boy, that there is a stimple drill.'"},
    {"role": "user",      "content": "A 'falbean' is a tool used to fasten, tighten, or spin/rotate things. "
                                     "Use both 'stimple' and 'falbean' in a sentence."},
]

print("STRONG — few-shot (definitions given as prior turns)")
print("=" * 60)
show(ask(few_shot_examples))

STRONG — few-shot (definitions given as prior turns)


The stimple falbean tightened the loose screw in no time.

In [6]:
# ── Part B: consulting classification (in-scope / out-of-scope) ───────────────
#
# Our convention: ACME's current engagement covers analytics and reporting only.
# Anything touching infrastructure, security, or HR is out of scope.
# The model can't infer this convention from language alone — it needs examples.

SCOPE_INSTRUCTIONS = (
    "You are a project scoping assistant for ACME Analytics. "
    "Classify each client requirement as IN_SCOPE or OUT_OF_SCOPE, "
    "and give a one-sentence reason. "
    "Reply with JSON: {\"classification\": \"IN_SCOPE\"|\"OUT_OF_SCOPE\", \"reason\": \"...\"}."
)

scope_examples = [
    {"role": "user",      "content": "Build a sales dashboard showing revenue by region."},
    {"role": "assistant", "content": '{"classification": "IN_SCOPE", "reason": "Revenue dashboards are core to ACME\'s analytics mandate."}'},
    {"role": "user",      "content": "Migrate the client\'s on-premise servers to AWS."},
    {"role": "assistant", "content": '{"classification": "OUT_OF_SCOPE", "reason": "Infrastructure migration falls outside analytics and reporting."}'},
    {"role": "user",      "content": "Create an automated report emailed to the CFO every Monday."},
]

resp = ask(scope_examples, instructions=SCOPE_INSTRUCTIONS)
raw  = resp.output_text
parsed = json.loads(raw)

print(f"Classification: {parsed['classification']}")
print(f"Reason:         {parsed['reason']}")
print()
print("Raw model output:", raw)

Classification: IN_SCOPE
Reason:         Automating and scheduling a recurring financial report for distribution to the CFO is a standard analytics deliverable within ACME's reporting capabilities.

Raw model output: {"classification":"IN_SCOPE","reason":"Automating and scheduling a recurring financial report for distribution to the CFO is a standard analytics deliverable within ACME's reporting capabilities."}


The model learned your classification convention from two examples — no fine-tuning, no retraining. That's **in-context learning**.

Notice we're already asking for JSON by instruction here. In Task 4 we'll replace that instruction with a *guaranteed* schema — but starting with a plain ask is fine for simple cases.

## Task 3 — Chain of thought + reasoning budget

Chain of thought (CoT) prompts the model to *reason before answering*, making the thinking step explicit. The model was already doing some reasoning internally — CoT makes it visible and auditable.

Modern gpt-5-era models take this further with a **reasoning budget**: `effort='low'` gives a quick, lightweight reasoning step; `effort='medium'` or `'high'` spends more tokens thinking before replying. The tradeoff is real and measurable: more effort costs more tokens and takes longer.

We'll use the classic timezone trick problem and compare `effort=None` (no explicit reasoning step) vs. `effort='low'`, printing answers, reasoning-token usage, and latency for both so the tradeoff is concrete.

In [7]:
REASONING_PROBLEM = """
I need to get my car cleaned. The car wash is only 50 meters away. Should I walk or drive?
"""

# gpt-5+ / o-series accept reasoning.effort; chat models (e.g. gpt-4o-mini) do not.
# On those, we fall back to the classic CoT prompt trick so the comparison still runs.
from helpers.llm_call import supports_reasoning_effort

SUPPORTS_EFFORT = supports_reasoning_effort(MODEL)
if not SUPPORTS_EFFORT:
    print(f"ℹ️  {MODEL} does not support reasoning.effort — comparing plain vs CoT prompt instead.\n")


def timed_ask(prompt, *, effort=None, instructions=None):
    t0 = time.perf_counter()
    resp = ask(prompt, instructions=instructions, effort=effort)
    elapsed = time.perf_counter() - t0
    usage = resp.usage
    details = getattr(usage, "output_tokens_details", None)
    reasoning_tokens = getattr(details, "reasoning_tokens", "n/a") if details else "n/a"
    return resp, elapsed, reasoning_tokens, usage.output_tokens, usage.total_tokens


runs = [
    ("None  (no explicit reasoning)", {"effort": None}),
    (
        "'High' (reasoning)" if SUPPORTS_EFFORT else "CoT prompt ('Let's think step by step.')",
        {"effort": "high"} if SUPPORTS_EFFORT else {
            "effort": None,
            "instructions": "Think carefully before answering. Let's think step by step.",
        },
    ),
]

for label, kwargs in runs:
    resp, elapsed, r_tok, out_tok, total = timed_ask(REASONING_PROBLEM, **kwargs)
    print("=" * 60)
    print(f"effort={label}")
    print(f"  reasoning tokens : {r_tok}")
    print(f"  output tokens    : {out_tok}")
    print(f"  total tokens     : {total}")
    print(f"  latency          : {elapsed:.1f}s")
    print()
    show(resp)
    print()


effort=None  (no explicit reasoning)
  reasoning tokens : 640
  output tokens    : 909
  total tokens     : 940
  latency          : 6.3s



Quick question: do you mean “should I walk there and leave the car at home” or “should I drive the car the 50 m to the car wash”? The answer differs.

- If you need to bring the car into the wash (it’s not a drop-off service), you obviously must drive it those 50 m.
- If you were thinking of walking there to drop off keys, pay, or ask about service while leaving the car at home, walk — it’s only ~30–60 seconds on foot and will save fuel, emissions and engine wear.
- If you’re deciding whether to drive the car itself the short distance to the wash: it’s fine and sometimes necessary, but short cold runs are inefficient and add wear. If you can combine the trip with other errands or use a drop-off/mobile service, that’s better for fuel and emissions.

Also consider weather, mobility, safety, whether you must transport items, and parking. If you tell me which situation applies, I’ll give a specific recommendation.


effort='High' (reasoning)
  reasoning tokens : 1920
  output tokens    : 2154
  total tokens     : 2185
  latency          : 13.0s



Short answer: If the car itself needs to go into the wash, drive it. If you just need to pop over on foot (to pay, book, or use a nearby service) then walk.

Things to consider:
- 50 m is tiny (≈1 minute on foot). Walking is greener, zero fuel/emissions, and avoids extra engine wear from very short cold starts.
- Driving 50 m usually takes longer once you factor starting, maneuvering and parking, and for petrol/diesel cars short trips are inefficient and can increase wear.
- If it’s a drive-through or the wash requires the car to be driven onto a bay, you have to drive it.
- Weather, mobility, what you’re carrying, or whether the wash offers pick‑up/drop‑off or mobile valeting can change the choice.

Want a specific recommendation for your situation? Tell me whether you mean bringing the car into the wash or just walking to the location.

**What to look for:**

- The catch is that **the car itself is what needs washing** — so you have to *drive*. Walking the 50 meters leaves your car at home, defeating the purpose. The "only 50 meters away" detail is a distractor designed to tempt a quick answer of "just walk, it's close."
- With `effort=None`, the model may anchor on the short distance and answer "walk" without noticing the car has to be *at* the car wash. With a higher reasoning effort, the explicit reasoning step catches the dependency more reliably and answers "drive."
- The token counts above show the cost difference. In production, **tune the effort to the task**: low for straightforward calls, higher only for problems that genuinely require multi-step reasoning.

> **Chain-of-thought prompt trick:** If you're using a standard (non-reasoning) model or API, you can elicit similar behavior by appending `"Let's think step by step."` to your user message. This prompts the model to reason in its output text before giving a final answer.

## Task 4 — Structured output enforcement

In Task 2 we asked the model politely to return JSON. That works most of the time — but "most of the time" isn't good enough when downstream code calls `json.loads()`.

**Structured Outputs** guarantee the response matches a schema you define. The SDK validates at the API layer and retries automatically if the model drifts. You define the schema once as a Pydantic model; the SDK generates the JSON schema that gets sent to the API.

We'll use a realistic scenario: extracting a structured engagement summary from a client briefing.

In [8]:
from pydantic import BaseModel
from typing import List, Literal

# ── Define the schema ────────────────────────────────────────────────────────
class EngagementSummary(BaseModel):
    client_name:    str
    engagement_type: Literal["analytics", "advisory", "implementation", "other"]
    key_objectives: List[str]     # up to 3 bullets
    risk_flag:      bool          # True if the briefing mentions risk or concerns
    one_liner:      str           # executive summary in one sentence

# ── Print the generated JSON schema so learners see the wire contract ─────────
import json as _json
print("JSON schema sent to the API:")
print(_json.dumps(EngagementSummary.model_json_schema(), indent=2))

JSON schema sent to the API:
{
  "properties": {
    "client_name": {
      "title": "Client Name",
      "type": "string"
    },
    "engagement_type": {
      "enum": [
        "analytics",
        "advisory",
        "implementation",
        "other"
      ],
      "title": "Engagement Type",
      "type": "string"
    },
    "key_objectives": {
      "items": {
        "type": "string"
      },
      "title": "Key Objectives",
      "type": "array"
    },
    "risk_flag": {
      "title": "Risk Flag",
      "type": "boolean"
    },
    "one_liner": {
      "title": "One Liner",
      "type": "string"
    }
  },
  "required": [
    "client_name",
    "engagement_type",
    "key_objectives",
    "risk_flag",
    "one_liner"
  ],
  "title": "EngagementSummary",
  "type": "object"
}


In [9]:
CLIENT_BRIEFING = """
The client is Northfield Medical Group, a regional hospital network.
They need help building an analytics platform to track patient-outcome KPIs
across their five hospitals. Key goals are real-time dashboards for clinical
leadership, a quarterly board-ready report, and a data-quality audit of the
existing EHR exports. They flagged a concern about HIPAA compliance for any
cloud-hosted components.
"""

EXTRACTION_INSTRUCTIONS = (
    "You are an ACME engagement analyst. "
    "Extract a structured summary from the client briefing."
)

# ── Parse into a typed Pydantic object ───────────────────────────────────────
# structured() asks for native schema-constrained decoding and validates the reply, so
# you get a real EngagementSummary back rather than a string you have to trust.
summary: EngagementSummary = structured(
    CLIENT_BRIEFING,
    EngagementSummary,
    instructions = EXTRACTION_INSTRUCTIONS,
)

print(f"Client:           {summary.client_name}")
print(f"Type:             {summary.engagement_type}")
print(f"Risk flag:        {summary.risk_flag}")
print(f"One-liner:        {summary.one_liner}")
print("Objectives:")
for obj in summary.key_objectives:
    print(f"  • {obj}")

Client:           Northfield Medical Group
Type:             analytics
Risk flag:        True
One-liner:        Develop an HIPAA-compliant analytics platform to deliver real-time patient-outcome dashboards, quarterly board reports, and a data-quality audit of EHR exports.
Objectives:
  • Build real-time dashboards for clinical leadership to track patient-outcome KPIs across five hospitals
  • Produce a quarterly board-ready patient-outcome report
  • Perform a data-quality audit of existing EHR exports


`summary` is a real Python object — fully typed, IDE-autocompleted, immediately usable by downstream code. No `json.loads`, no KeyError.

In production this pattern replaces fragile regex parsing and brittle prompt instructions. The schema becomes the contract between the prompt layer and the application layer — any engineer on the team can read it and know exactly what to expect.

## Task 5 — Leveraging manually retrieved context

By default, the model answers from training data — which cuts off at a point in the past and knows nothing about your client, your internal documents, or your project specifics.

The simplest fix is to **paste the relevant document into the instructions**. The model reads it and answers from what's there. This is "RAG by hand" — retrieval-augmented generation without the retrieval pipeline.

We'll use the shared EU AI Act summary from the repo. Watch the difference between the no-context answer and the grounded answer that has the course document pasted into the prompt.

In [10]:
EU_AI_ACT_SUMMARY = (Path("..") / "data" / "eu_ai_act_summary.md").read_text()
CONTEXT_Q = (
    "What are the EU AI Act's four risk tiers, "
    "and when do most high-risk obligations apply?"
)

# ── Weak: no context ─────────────────────────────────────────────────────────
print("=" * 60)
print("WEAK — no context (model answers from training data only)")
print("=" * 60)
show(ask(CONTEXT_Q))

WEAK — no context (model answers from training data only)


The EU AI Act divides AI systems into four risk tiers:

1. Unacceptable risk (prohibited) — practices judged to unduly threaten safety or fundamental rights and therefore banned (e.g., certain manipulative or discriminatory social‑scoring uses).  
2. High risk — AI that poses substantial risk to health, safety or fundamental rights (systems used in critical infrastructure, education/occupational screening, employment, biometric ID, law‑enforcement, migration/control of borders, administration of justice, or any AI that is a safety component of a product covered by existing EU product‑safety laws).  
3. Limited (or transparency) risk — systems that require specific transparency obligations so users can make informed choices (e.g., chatbots, deepfakes, some emotion‑recognition tools).  
4. Minimal/no risk — most AI applications; no specific EU AI Act rules beyond general law.

When do most high‑risk obligations apply?
- The bulk of high‑risk duties (risk‑management system, data governance, technical documentation, conformity assessment, logging/traceability, human oversight, robustness and cybersecurity, CE‑type marking/registration) must be fulfilled before the system is placed on the market or put into service and maintained throughout the system’s lifecycle.  
- Obligations fall primarily on the AI provider (and in many cases the deployer/operator), with continuing compliance and documentation requirements while the system is used.

In [11]:
# ── Strong: summary in instructions ──────────────────────────────────────────
GROUNDED_INSTRUCTIONS = (
    "You are an EU AI Act briefing assistant. "
    "Answer questions using ONLY the summary below. "
    "If the answer is not in the summary, say so explicitly.\n\n"
    + EU_AI_ACT_SUMMARY
)

print("=" * 60)
print("STRONG — summary in instructions (grounded answer)")
print("=" * 60)
show(ask(CONTEXT_Q, instructions=GROUNDED_INSTRUCTIONS))

STRONG — summary in instructions (grounded answer)


The Act uses four risk tiers:

- Unacceptable risk — prohibited (Article 5): short list of banned practices (e.g., manipulative/deceptive techniques, exploitation of vulnerabilities, social scoring by public authorities, untargeted facial-image scraping, emotion recognition in workplaces/schools, real-time remote biometric ID in public spaces with narrow law-enforcement exceptions).
- High risk (Annex III + safety components): permitted but heavily regulated (risk management, data governance, documentation and logging, transparency and human oversight, accuracy/robustness/cybersecurity, conformity assessment and registration).
- Limited risk — transparency (Article 50): interaction/content systems must disclose they are AI (chatbots, labelled synthetic/manipulated media).
- Minimal risk: everything else (no obligations).

Most high-risk obligations apply on 2 August 2026.

The grounded answer can cite the exact risk tiers and timeline dates from the summary.

This pattern works immediately for **any document you can fit in the context window**. For larger corpora — thousands of documents, or documents that change frequently — you would build a retrieval pipeline (vector search, chunking, re-ranking) to select the right passages programmatically. That's the production version of what you just did by hand.

## Task 6 — Self-refine

A single prompt is rarely perfect on the first pass. **Self-refine** is a two-call loop:

1. Ask the model for a draft.
2. Ask the model to *critique the draft against explicit criteria* and produce an improved version.

You get the quality benefit of a human review cycle without the wait — and you can scale it to as many passes as you need.

We'll use a realistic AI governance task: drafting a client-facing recommendation for a product team that may be building a high-risk AI system.

In [12]:
DRAFT_INSTRUCTIONS = (
    "You are an AI governance consultant. "
    "Write a client-facing recommendation on the question below. "
    "Use the EU AI Act summary as your only factual source.\n\n"
    + EU_AI_ACT_SUMMARY
)

DRAFT_Q = "What should a product team do if its AI system may fall into a high-risk EU AI Act category?"

# ── Pass 1: draft ────────────────────────────────────────────────────────────
draft_resp = ask(DRAFT_Q, instructions=DRAFT_INSTRUCTIONS)
draft = draft_resp.output_text

print("=" * 60)
print("PASS 1 — draft")
print("=" * 60)
show(draft_resp)

PASS 1 — draft


Short answer
If your system may be high‑risk under the EU AI Act, treat it as high priority: you must put in place a documented risk-management and compliance programme now (data governance, technical documentation and logging, human oversight, robustness and cybersecurity testing), prepare for a conformity assessment and registration, and be ready for the Act’s timelines (most high‑risk obligations apply 2 August 2026). The provider, deployer, importer and distributor can all have obligations, and the law applies whenever the system’s outputs are used in the EU.

Recommended next steps (practical, prioritized)

1) Immediate triage (days–weeks)
- Confirm classification: determine whether the system is listed in Annex III (biometrics; critical infrastructure; education; employment/worker management; access to essential public/private services; law enforcement; migration/border control; administration of justice) or is a safety component of a regulated product. If unsure, treat it as high‑risk while you investigate.
- Map roles and responsibility: identify who in your organisation is the “provider” and “deployer” for the EU AI Act, and identify legal/compliance owners, product owners, security lead and an executive sponsor.

2) Build a compliance plan (weeks)
- Create a project plan aligned to the Act’s timeline:
  - Most high‑risk obligations effective 2 Aug 2026 — aim to complete core controls well before that date.
  - If your system is a safety component of a regulated product, additional obligations apply from 2 Aug 2027.
- Assign deliverables, owners, and dates for each compliance requirement below.

3) Implement required controls (months)
- Risk management system: set up a documented, proactive risk‑management process for the AI lifecycle (risk identification, assessment, mitigation, monitoring, review).
- Data governance and quality: document training, validation and test datasets; put in place data quality and provenance checks; processes to reduce bias and ensure representativeness.
- Technical documentation and logging: produce system technical documentation describing intended use, architecture, datasets summary, performance metrics, limitations, and keep runtime logs for traceability and incident analysis.
- Accuracy, robustness and cybersecurity testing: define and run testing plans for performance, robustness to faults and attacks, and cybersecurity controls. Capture results as evidence for conformity assessment.
- Human oversight: design and document measures that enable human oversight appropriate to the risk (roles, interfaces, instructions, escalation paths) so the human can meaningfully supervise and intervene.
- Transparency: ensure required disclosures (e.g., system capabilities/limitations) are available to users and deployers as applicable.

4) Prepare for conformity assessment and registration
- Plan for and budget the conformity assessment (internal or notified body as applicable) and for registration of the high‑risk system in the EU database. Collate the technical file and testing evidence the assessment will require.
- Maintain an internal audit trail and version control for releases; keep documentation up to date as the system evolves.

5) Post‑market obligations and incident readiness
- Put post‑market monitoring in place (ongoing performance monitoring, user feedback, incident detection).
- Prepare incident response procedures and reporting processes (the Act imposes obligations such as reporting and maintaining logs).
- Keep a process for corrective actions and re‑assessment when the system changes.

6) Governance, training and assurance
- Establish governance and oversight (steering committee or board updates).
- Train product, engineering and operations teams on the new obligations and on how to operate within the risk‑management and human‑oversight framework.
- Schedule periodic compliance reviews and readiness checks ahead of the 2026 deadline.

Key practical deliverables to produce now
- Classification memo: why you believe the system is (or isn’t) high‑risk.
- Risk‑management plan and register of identified risks/mitigations.
- Data‑governance checklist and dataset inventory/quality reports.
- Technical documentation package and logging policy.
- Human‑oversight design spec and user instructions.
- Test plans and evidence for accuracy, robustness and cybersecurity.
- Conformity‑assessment readiness folder.

Risks and enforcement to note
- The Act applies extraterritorially: it covers systems whose outputs are used in the EU even if your company is outside the EU.
- Fines for breaching AI Act obligations are significant (for most other obligations up to €15 million or 3% of worldwide turnover; for misleading or incorrect information up to €7.5 million or 1%).
- Prohibited practices take effect 2 Feb 2025 — ensure system behaviour does not fall into the banned categories.

Final recommendations
- Don’t delay: begin the classification and set up your risk‑management and documentation processes immediately.
- Treat compliance as product lifecycle work: bake data governance, testing, logging and human‑oversight into your development and release processes rather than trying to bolt them on at the end.
- Prepare for conformity assessment and registration well before 2 August 2026.
- Engage legal/compliance early to validate classification and to help scope the conformity process.

If you want, I can: a) draft a one‑page classification checklist tailored to your product; b) outline a project plan and milestone calendar to achieve conformity readiness by 2 Aug 2026; or c) create templates for the technical documentation and risk register. Which would you prefer?

In [13]:
# ── Pass 2: critique + revision ───────────────────────────────────────────────
REFINE_INSTRUCTIONS = (
    "You are a senior editor reviewing an AI governance recommendation. "
    "Evaluate the draft against these criteria:\n"
    "  1. Conciseness — is it under 150 words?\n"
    "  2. Evidence — does every claim cite a specific fact from the summary?\n"
    "  3. Action — does it end with a single, clear recommendation?\n\n"
    "First produce a short critique (2–3 sentences). "
    "Then produce the revised recommendation under the heading 'REVISED:'.\n\n"
    "SUMMARY:\n" + EU_AI_ACT_SUMMARY
)

refine_input = f"DRAFT:\n{draft}\n\nPlease critique and revise."
refine_resp  = ask(refine_input, instructions=REFINE_INSTRUCTIONS)

print("=" * 60)
print("PASS 2 — critique + revision")
print("=" * 60)
show(refine_resp)

PASS 2 — critique + revision


Critique: The draft is thorough and practical but exceeds the 150-word conciseness requirement, does not cite specific facts from the provided summary for each claim, and ends with multiple suggested actions rather than a single, clear recommendation. It should be shortened, each assertion tied to a summary fact, and conclude with one concrete next step.

REVISED:
If the system may be high‑risk, begin compliance work now (Summary: "High risk... Providers must run a risk management system... pass a conformity assessment and register the system"). Implement data governance, technical documentation and logging, human oversight, and accuracy/robustness and cybersecurity testing (Summary: "Providers must... ensure data governance and quality, keep technical documentation and logs, provide... human oversight, and meet accuracy, robustness, and cybersecurity standards"). Remember the Act is extraterritorial (Summary: "it reaches any system whose output is used in the EU") and most high‑risk obligations apply from 2 Aug 2026 (Summary: "Most high‑risk obligations effective 2 Aug 2026"). Recommendation: Immediately classify the system against Annex III and, if plausibly high‑risk, start a documented risk‑management and conformity‑readiness programme. (Summary: "Annex III... biometrics; critical infrastructure; education; employment/worker management; access to essential...")

Self-refine is powerful because the **critique criteria are explicit and auditable** — you can read them, adjust them, and hand them to a stakeholder as the definition of "good."

In production, multi-pass self-refine loops are often the fastest way to lift output quality before you invest in fine-tuning. Agent frameworks (LangGraph, CrewAI) encode these loops as durable workflows — but the underlying pattern is exactly what you just built.

## Task 7 — Meta-prompting

The patterns we've covered so far require you to *write* the prompt. Meta-prompting flips that: you ask the model to **write or improve a prompt for you**.

This is surprisingly powerful for recurring tasks where you want to nail the prompt once and reuse it. Give the model a task description, some examples of bad output, and your quality criteria — it returns a reusable prompt. Then run that generated prompt to confirm it works.

We'll generate a prompt for a recurring ACME task: summarizing client meeting notes into a structured action-item list.

In [14]:
META_INSTRUCTIONS = (
    "You are a prompt-engineering expert. "
    "Write a system prompt (instructions string) for the task described below. "
    "The prompt should be clear, concise, and produce consistent outputs "
    "when passed to a gpt-5-series model via the Responses API.\n\n"
    "Wrap your final prompt in <PROMPT> ... </PROMPT> tags so it can be extracted."
)

TASK_DESCRIPTION = """
TASK: Extract action items from ACME consulting meeting notes.

Good output:
  - A numbered list of action items
  - Each item names the owner (person or team), the action, and the due date if mentioned
  - Items are ordered by due date (soonest first, undated items last)

Bad output we've seen:
  - Prose paragraphs instead of a list
  - Items without an owner
  - Invented due dates not mentioned in the notes
"""

meta_resp = ask(TASK_DESCRIPTION, instructions=META_INSTRUCTIONS)

print("=" * 60)
print("Generated prompt:")
print("=" * 60)
show(meta_resp)

Generated prompt:


<PROMPT>
You are an action-item extractor for ACME consulting meeting notes. Given meeting notes, output only a clean, ordered list of action items.

Rules:
- Output format: a numbered list. Each item must follow this pattern:
  N. Owner — Action — Due: <due date or "None specified">
- Every item must name an owner (person or team). If the notes do not specify an owner, use "Unassigned" as the owner.
- Include only action items that are stated or clearly assigned in the notes. Do NOT invent new actions, owners, or due dates.
- Due dates: copy the due date text exactly as written in the notes. Do not create or infer dates. If no due date is mentioned, use "None specified".
- Ordering: list items by due date, earliest first. Place items with no clear calendar date (including vague or relative phrases like "ASAP", "next Friday", "by end of month" unless a calendar date is explicitly given) after all explicitly dated items. Do not convert or infer relative dates into calendar dates.
- If multiple items share the same due date, their relative ordering may be arbitrary.
- Output only the numbered list. Do not add explanations, summaries, or the original notes.
- If there are no action items, output exactly: "No action items found."

Example:
1. Marketing Team — Prepare Q3 campaign plan — Due: July 15, 2026
2. Unassigned — Compile stakeholder feedback from meeting — Due: None specified

Now extract action items from the provided meeting notes following the rules above.
</PROMPT>

In [15]:
import re

# ── Extract and run the generated prompt ─────────────────────────────────────
raw_meta = meta_resp.output_text
match = re.search(r"<PROMPT>(.*?)</PROMPT>", raw_meta, re.DOTALL)
generated_prompt = match.group(1).strip() if match else raw_meta.strip()

MEETING_NOTES = """
Date: 10 July 2026  —  ACME / Northfield Medical Group sync

Maria (ACME PM) to send the revised project charter by Friday.
James (client lead) will get sign-off from the CMO before end of month.
The data team needs to audit EHR export fields before the dashboard design can start
— no firm date yet, but blocking the next milestone.
Everyone agreed to a weekly 30-min standup starting next Tuesday at 9 AM ET.
"""

print("=" * 60)
print("Generated prompt used as instructions → action-item extraction:")
print("=" * 60)
show(ask(MEETING_NOTES, instructions=generated_prompt))

Generated prompt used as instructions → action-item extraction:


1. Maria — Send the revised project charter — Due: by Friday
2. James — Get sign-off from the CMO — Due: before end of month
3. Data team — Audit EHR export fields before the dashboard design can start — Due: no firm date yet, but blocking the next milestone
4. Everyone — Attend weekly 30-min standup starting next Tuesday at 9 AM ET — Due: starting next Tuesday at 9 AM ET

The model wrote a prompt — and it works. That's meta-prompting: **using the model as a prompt engineer**.

This is especially useful for:
- Rapidly drafting a starting point for a new task
- Systematically improving prompts given failure examples
- Generating prompt variants for A/B testing

In a mature AI platform, meta-prompting feeds into a prompt-management system where generated prompts are version-controlled, tested against evals, and deployed like code.

## 🏗️ Activity — Stack the patterns

Each pattern above stands on its own. The real leverage comes from **combining them**.

Your task: write a single prompt for the EU AI Act high-risk question that uses at least three patterns at once. A suggested combination:
- **Persona** — set a specific voice in `instructions`
- **Retrieved context** — include the summary
- **Structured output** — parse the recommendation into a Pydantic model with a decision (`proceed`/`redesign`/`pause`), a rationale, and a list of risks

Then compare it against the naive one-liner from Task 1.

In [16]:
from pydantic import BaseModel
from typing import Literal, List

# ── Define the output schema ──────────────────────────────────────────────────
class HighRiskPlan(BaseModel):
    decision:  Literal["proceed", "redesign", "pause"]
    rationale: str             # 2-3 sentences grounded in the summary
    risks:     List[str]       # up to 3 risks

# ── Your stacked prompt ───────────────────────────────────────────────────────
# TODO: fill in a persona, include the EU_AI_ACT_SUMMARY, and parse into HighRiskPlan
STACKED_INSTRUCTIONS = (
    # Your persona here
    "You are an AI governance lead. Give a direct, evidence-based recommendation. "
    # Your retrieved context here
    "Use ONLY the summary below.\n\n" + EU_AI_ACT_SUMMARY
)

decision: HighRiskPlan = structured(
    "Should a product team proceed if its AI system may be high-risk under the EU AI Act?",
    HighRiskPlan,
    instructions = STACKED_INSTRUCTIONS,
)
print(f"Decision:  {decision.decision.upper()}")
print(f"Rationale: {decision.rationale}")
print("Risks:")
for r in decision.risks:
    print(f"  • {r}")

Decision:  REDESIGN
Rationale: The EU AI Act imposes heavy, specific obligations on high‑risk systems (risk management, data governance and quality, technical documentation and logs, transparency and human oversight, accuracy/robustness and cybersecurity) plus conformity assessment and registration. Non‑compliance carries substantial penalties (up to €15 million or 3% of worldwide turnover) and the law applies extraterritorially to systems whose output is used in the EU. Where feasible, redesigning to avoid a high‑risk classification (or to fall into a lower risk tier) is the most effective way to reduce regulatory burden, cost, and enforcement risk. If redesign is infeasible, the team must instead pause and invest in a full compliance program (risk management, documentation, tests, and conformity assessment) before deployment.
Risks:
  • Regulatory non‑compliance leading to fines up to €15 million or 3% of worldwide turnover
  • Significant development and operational cost to meet hig

## What We Just Built vs. What's In Production

| What we built | Production equivalent |
|---------------|----------------------|
| Hand-written few-shot examples in the prompt | Curated example stores with dynamic selection (nearest-neighbor retrieval of the most relevant examples) |
| Manual document paste into `instructions` | Retrieval pipeline: vector search over a document corpus, chunking, re-ranking, context assembly |
| Two-call self-refine loop | Reflection agents and durable multi-step workflows (LangGraph, Vercel Workflow, CrewAI) |
| Pydantic model as the output contract | Schema registries, function-calling specs, and validation middleware shared across the team |
| One persona string per call | Versioned system-prompt templates deployed like code; A/B testing across variants |
| Reading output quality by eye | Automated evals with judges and CI gates — the subject of Demo 03 |
| Reasoning effort set by hand | Dynamic effort routing: cheap calls default to `low`, fall back to `high` when confidence is low |

---

## 🚀 Advanced Build

**Build a prompt regression set.** Pin down a handful of inputs with known-correct answers (e.g. three more scope-classification cases from Task 2, or briefings with an expected `engagement_type` from Task 4). Run your prompt across all of them and assert the structured output matches what you expect. Now any prompt edit either passes or fails the set — you've turned "looks good to me" into a repeatable check, using nothing but the structured-output pattern you already built. (Demo 03 generalizes this into full LLM-judged evals.)

**Select few-shot examples dynamically.** Instead of hard-coding two examples, embed a library of labeled examples and retrieve the most similar ones to each new input at call time. This is the pattern that powers in-context learning at scale.

**Tune the reasoning budget per task.** Profile every prompt in your application: log `usage.output_tokens_details.reasoning_tokens` and latency. Set `effort='none'` for simple lookups, `effort='low'` for straightforward reasoning, and reserve `effort='high'` only for the tasks that genuinely need it.